
# Experiment v13b: a validated decision rule (response to the supervisor's review)

Builds on v9b. Five changes, all written **before** running:

1. **Uncertainty in Q1.** The teacher is called better than the shared score only if a paired bootstrap (2,000 resamples of the local women)
   says so with probability ≥ c, for c ∈ {point estimate, 0.80, 0.90, 0.95}. The truth is known from the 9,500-woman test set, so the
   error rate of Q1 is measured.
2. **ESS threshold.** Thresholds 10, 20, 30, 40, 50% are compared.
3. **Gain prediction.** Predicted vs observed gain on *all* scenarios with MAE, RMSE, bias and rank correlation, for the framework
   (true risk known) and for the practical estimate available to the unit, raw and **bias-corrected** (corrected = estimate on the targeted
   pool minus the same estimate on an untargeted, donor-like pool, which should be zero and therefore measures the bias).
4. **Design vs held-out scenarios.** The rule's parameters (c, use of the gain question, ESS threshold) are chosen on the *design*
   scenarios only (the v9 grid) and then evaluated, unchanged, on *held-out* scenarios never used to build the rule:
   two new shift variables (age, in the score; one omitted predictor) × {0.6, 0.8, 0.95}, and the v9 shift variables × {0.8, 0.95}.
5. **Ablation and explainability.** M0 local only · M1 shared model + updating · M2 untargeted synthetic · M3 targeted synthetic ·
   M4 full framework (selected rule). Supplementary: standardized coefficients and feature importance (for a logistic score, the mean
   |SHAP| of feature j on the logit scale is |β_j|·E|x_j − x̄_j|) of each model compared with the receiver's best simple score.

Ties in design regret are broken in a fixed, pre-registered order: no gain question first, then the simplest Q1, then the median of the tied ESS thresholds.\n\nSensitivity runs use the same notebook with `TEACHER_N = 1000` (weaker teacher) or `N_LOCAL = 100 / 400`.

## Pre-registered hypotheses
- **H13.1** The bias-corrected practical estimate has smaller |bias| and smaller MAE than the raw estimate.
- **H13.2** The bootstrap Q1 (c = 0.80 or 0.90) makes fewer false "teacher better" calls than the point estimate, without higher regret.
- **H13.3** The rule chosen on the design scenarios keeps, on the held-out scenarios, a mean regret lower than every fixed strategy.
- **H13.4** On the held-out scenarios M4 has the highest mean gain over M1 and the best worst case among M1–M4.
- **H13.5** Where a gain is predicted (Γ_upd(n) ≥ 2), the targeted synthetic score (M3) is closer in direction (cosine of standardized
  coefficients) and in feature-importance ranking to the receiver's best simple score than the shared model.


---

# Experiment v9b: can a small unit choose the right transfer method on its own?

v8 showed **what** to choose given the true expected gain. In practice a small unit does not know the truth. Here the unit decides **using only what it has**: the donor's shared score, the donor's soft-labelled synthetic pool (the teacher's probabilities), and its own 200 patients.

**Estimates made by the receiver (no access to the true risk):**
1. `est_G` and `est_Gupd_n`: the same formulas as the theoretical gains, with the teacher's probabilities on the targeted pool used in place of the true risk.
2. **Teacher check:** log-loss of the teacher vs the shared score on the 200 local patients. A teacher that is not better than the score cannot transfer anything.
3. `ess_don`: effective sample size of the density-ratio weights on the donor's patients. Low ESS = strong shift, where reweighting is fragile.

## Pre-registered decision rule
- If the teacher is not better than the shared score on local data, **or** `est_Gupd_n` < 2 (nats × 1000) → **share the model and update it** (closed testing).
- Otherwise, if `ess_don` < 0.30 (strong shift) → **targeted synthetic data**.
- Otherwise → **direct reweighting at the donor**.

## Evaluation
For every repetition, the method chosen by the rule is compared with the method that turned out best on the test set, among: updated model sharing, direct reweighting, targeted synthetic data.
- **Regret** = loss of the chosen method minus loss of the best method.
- The rule's regret is compared with three fixed strategies (always share, always reweight, always synthetic).
- We also report how well `est_Gupd_n` tracks the true $\Gamma_{\rm upd}(n)$.



This notebook extends v7 (real outcome, short clinical score) with the comparisons requested in the review. All arms are evaluated on the same real test patients with the same protocol.

| Arm | What it tests |
|---|---|
| updated model sharing (closed testing) | practical baseline |
| `IW_donor` | **direct reweighting at the donor**: the receiver sends its density-ratio classifier (trained on its covariates vs the donor's synthetic covariates); the donor refits its simple model on its **real** patients with these weights. Can the gain be obtained without generating data? |
| `IW_donor_ct` | the same, then closed-testing update at the receiver |
| `local_distill` | the donor's teacher is transferred; the receiver distils it on its own covariates. Is there a simpler transfer of knowledge? |
| `C_target` | targeted synthetic data (ours). Does generation add value beyond the alternatives? |
| `IW_donor_exact`, `C_exact_ratio` | the same two methods with the **true** density ratio (known by design) — isolates the effect of density-ratio estimation |

## Pre-registered hypotheses (cells where the theory predicts a gain)
- **H1 (asymptotic equivalence):** with the exact ratio, reweighting and targeted synthetic data both target $h_R^\ast$; their difference is below 25% of $\Gamma$.
- **H2 (finite-sample equivalence, n = 200):** the same holds with estimated ratios.
- **H3 (density-ratio error):** reported, not predicted: the loss from estimating the ratio, for each carrier.
- **H4 (finite-sample prediction):** the excess risk actually reached by closed testing with $n$ = 200 local patients is predicted better by its finite-sample counterpart (the population donor model updated by a closed test on $n$ simulated patients) than by the asymptotic recalibration used in $\Gamma_{\rm upd}$. This addresses the cell where $\Gamma_{\rm upd}$ under-predicted the gain.
- **Clinical:** calibration slope, calibration-in-the-large and net benefit at 10/15/20% risk thresholds.

**Operational differences (not tested, reported):**
- **Reweighting** needs two rounds per receiver: the donor sends covariates, the receiver returns a classifier, the donor refits.
- **Targeted synthetic data** need one transfer: the same pool serves every receiver, and each receiver targets it locally.
- **Privacy:** the disclosure risk of a teacher-labelled synthetic pool, such as membership inference, is outside the scope of this study.

In [ ]:
%pip install -q sdv xgboost statsmodels ucimlrepo

## 1. Configuration

In [ ]:
DATASET = "natality"              # "natality" or "cdc_diabetes"
NAT_URL, NAT_READ, NAT_N = "https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Datasets/DVS/natality/Nat2023us.zip", 0.02, 10**9
CSV_PATH, TARGET_COL, POSITIVE, DROP_COLS = "x.csv", None, None, []

if DATASET == "natality":
    SCORE = ["Age", "BMI", "nulliparous"]
    SHIFTS = {"BMI (in score)": "BMI", "education (omitted)": "education", "age (in score)": "Age", "prior births (omitted)": "prior_births"}
    POP_N = 5000
else:
    SCORE = ["Age", "BMI", "HighBP"]
    SHIFTS = {"BMI (in score)": "BMI", "general health (omitted)": "GenHlth", "age (in score)": "Age", "income (omitted)": "Income"}
    POP_N = 20000
SKEWS    = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
_OLD     = list(SHIFTS)[:2]; _NEW = list(SHIFTS)[2:]
DESIGN   = [(v, s) for v in _OLD for s in [0.5, 0.7, 0.9]]                                    # the v9 grid: used to choose the rule
HELDOUT  = [(v, s) for v in _NEW for s in [0.6, 0.8, 0.95]] + [(v, s) for v in _OLD for s in [0.8, 0.95]]
SCEN     = DESIGN + HELDOUT
N_LOCAL, N_DONOR, TEST_N, N_SYN, POOL_MULT = 200, 10000, 9500, 10000, 3
REF_SHARE = 0.20                  # random share of the data reserved for the reference model eta_hat
W_CLIP_Q, CART_MIN_LEAF, ALPHA_CT = 0.99, 5, 0.05

QUICK  = True                     # True: 2 repetitions. False: 20 repetitions.
N_REPS = 2 if QUICK else 20
SEED0  = 2026
TEACHER_N = None                  # sensitivity: 1000 = weaker teacher (trained on 1,000 donor women)
OUT_CSV = f"results_v13b_{DATASET}_{'quick' if QUICK else 'full'}{'' if N_LOCAL == 200 else f'_n{N_LOCAL}'}{'' if TEACHER_N is None else f'_t{TEACHER_N}'}.csv"
OUT_COEF = OUT_CSV.replace('.csv', '_coef.csv')
print(f"{DATASET} QUICK={QUICK} reps={N_REPS} score={SCORE} -> {OUT_CSV}")

## 2. Data and reference model

In [ ]:
import warnings, os, time, itertools
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd

NAT_FIELDS = {   # 1-based positions from the NCHS 2023 natality user guide -> python slices
    "MAGER": (74, 76), "BMI": (282, 286), "RF_PDIAB": (312, 313), "RF_PHYPE": (314, 315),
    "RF_GHYPE": (315, 316), "RF_EHYPE": (316, 317), "RF_PPTERM": (317, 318), "RF_INFTR": (324, 325),
    "CIG_0": (252, 254), "PRIORLIVE": (170, 172), "PRIORDEAD": (172, 174), "MEDUC": (123, 124),
    "DPLURAL": (453, 454), "PAY_REC": (435, 436)}

def _zip_lines(zpath):
    # The CDC archive uses Deflate64, which Python's zipfile cannot read; stream it through the system unzip tool instead.
    import zipfile, subprocess, shutil
    try:
        with zipfile.ZipFile(zpath) as z:
            name = [n for n in z.namelist() if n.lower().endswith(".txt")][0]
            with z.open(name) as f:
                yield from f
        return
    except NotImplementedError:
        pass
    if shutil.which("unzip"):
        cmd = ["unzip", "-p", zpath]
    elif shutil.which("7z"):
        cmd = ["7z", "e", "-so", zpath]
    else:
        raise RuntimeError("Install unzip first:  !apt-get -qq install unzip")
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, bufsize=1 << 20)
    yield from p.stdout
    if p.wait() != 0:
        raise RuntimeError(f"{cmd[0]} failed while reading {zpath}")

def load_natality():
    import urllib.request, zipfile, random
    cache = "natality2023_sample.csv"
    if not os.path.exists(cache):
        zpath = "Nat2023us.zip"
        if not os.path.exists(zpath):
            print("downloading the 2023 natality file (about 230 MB) ..."); urllib.request.urlretrieve(NAT_URL, zpath)
        rnd = random.Random(SEED0); rows = []
        for line in _zip_lines(zpath):
            if rnd.random() >= NAT_READ: continue
            s = line.decode("latin-1")
            rows.append([s[a:b] for a, b in NAT_FIELDS.values()])
        pd.DataFrame(rows, columns=list(NAT_FIELDS)).to_csv(cache, index=False)
        print(f"read {len(rows)} birth records -> {cache}")
    d = pd.read_csv(cache, dtype=str).apply(lambda c: c.str.strip())
    n0 = len(d)
    rf = ["RF_PDIAB", "RF_PHYPE", "RF_GHYPE", "RF_EHYPE", "RF_PPTERM", "RF_INFTR"]
    d = d[(d.DPLURAL == "1") & d[rf].isin(["Y", "N"]).all(axis=1)]
    num = {c: pd.to_numeric(d[c], errors="coerce") for c in ["MAGER", "BMI", "CIG_0", "PRIORLIVE", "PRIORDEAD", "MEDUC", "PAY_REC"]}
    ok = (num["BMI"] < 99) & (num["CIG_0"] < 99) & (num["PRIORLIVE"] < 99) & (num["PRIORDEAD"] < 99) \
         & num["MEDUC"].between(1, 8) & (d.RF_PHYPE == "N")      # women with chronic hypertension excluded
    d = d[ok]; num = {k: v[ok] for k, v in num.items()}
    parity = num["PRIORLIVE"] + num["PRIORDEAD"]
    X = pd.DataFrame({
        "Age": num["MAGER"], "BMI": num["BMI"],
        "nulliparous": (parity == 0).astype(int), "prior_births": parity.clip(upper=6),
        "pregestational_diabetes": (d.RF_PDIAB == "Y").astype(int),
        "previous_preterm": (d.RF_PPTERM == "Y").astype(int), "infertility_treatment": (d.RF_INFTR == "Y").astype(int),
        "cigarettes_before": num["CIG_0"].clip(upper=40), "education": num["MEDUC"]})
    y = ((d.RF_GHYPE == "Y") | (d.RF_EHYPE == "Y")).astype(int)
    out = X.assign(y=y.values).sample(min(NAT_N, len(X)), random_state=SEED0).reset_index(drop=True)
    print(f"natality: {n0} records read | {len(X)} singleton births, no chronic hypertension, complete data | {len(out)} kept")
    return out

def load_raw():
    if DATASET == "natality":
        return load_natality()
    if DATASET == "uci":
        from ucimlrepo import fetch_ucirepo
        d = fetch_ucirepo(id=863)
        X = d.data.features.copy()
        y_raw = d.data.targets.iloc[:, 0].astype(str).str.strip().str.lower()
        y = (y_raw == "high risk").astype(int)          # high risk vs low/mid risk
    else:
        df = pd.read_csv(CSV_PATH).drop(columns=DROP_COLS, errors="ignore")
        tcol = TARGET_COL or df.columns[-1]
        pos = POSITIVE if POSITIVE is not None else df[tcol].value_counts().idxmin()
        y = (df[tcol] == pos).astype(int); X = df.drop(columns=[tcol])
        print(f"Outcome column {tcol!r}, positive value {pos!r}")
    X = X.select_dtypes(include="number")
    X = X[X.columns[X.isna().mean() <= 0.30]]
    return X.assign(y=y.values)

def audit(df):
    if DATASET == "natality":   # registry data: identical rows are different women, so no de-duplication
        feats = [c for c in df.columns if c != "y"]
        print(f"features: {feats}\nprevalence of gestational hypertension/eclampsia: {df.y.mean():.3f}")
        return df.reset_index(drop=True)
    n0 = len(df); df = df.dropna(); n1 = len(df)
    df = df.drop_duplicates(); n2 = len(df)
    feats = [c for c in df.columns if c != "y"]
    conflict = df.groupby(feats)["y"].transform("nunique") > 1
    df = df[~conflict].reset_index(drop=True)
    print(f"rows: raw {n0} | complete {n1} | after removing exact duplicates {n2} "
          f"| after removing {int(conflict.sum())} conflicting rows {len(df)}")
    print(f"features: {feats}\nprevalence: {df.y.mean():.3f}")
    assert SHIFT_VAR in feats, f"SHIFT_VAR {SHIFT_VAR!r} is not a column"
    return df



def load_cdc():
    cache = "cdc_diabetes_binary.csv"
    if os.path.exists(cache):
        df = pd.read_csv(cache)
    else:
        from ucimlrepo import fetch_ucirepo
        d = fetch_ucirepo(id=891)
        df = d.data.features.copy(); df["y"] = d.data.targets["Diabetes_binary"].values
        df.to_csv(cache, index=False)
    df = df.astype(float); df["y"] = df["y"].astype(int)
    print(f"CDC Diabetes Health Indicators: {len(df)} adults, prevalence {df.y.mean():.3f}")
    return df.reset_index(drop=True)

DATA = load_cdc() if DATASET == "cdc_diabetes" else audit(load_raw())
FEATURES = [c for c in DATA.columns if c != "y"]


from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import log_loss
from xgboost import XGBClassifier

X_ALL = DATA[FEATURES].to_numpy(float)
Y_ALL = DATA.y.to_numpy()
S_IDX = [FEATURES.index(c) for c in SCORE]
expit = lambda u: 1 / (1 + np.exp(-u))

_perm = np.random.default_rng(SEED0).permutation(len(DATA))
REF = _perm[: int(REF_SHARE * len(DATA))]
IN_REF = np.zeros(len(DATA), bool); IN_REF[REF] = True
ETA_HAT = XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.9, verbosity=0, n_jobs=-1) \
    .fit(X_ALL[REF], Y_ALL[REF]).predict_proba(X_ALL)[:, 1]

HIGH, RULES = {}, {}
for name, v in SHIFTS.items():
    col = DATA[v].to_numpy(float); med = np.median(col[~IN_REF])
    h = col > med
    if h[~IN_REF].mean() < 0.3: h = col >= med
    HIGH[name] = h; RULES[name] = (FEATURES.index(v), med, bool(h[~IN_REF].mean() != (col[~IN_REF] > med).mean()))
    print(f"shift '{name}': share above the median = {h[~IN_REF].mean():.2f}")
_rest = ~IN_REF
_ri = np.flatnonzero(_rest); _h = min(20000, len(_ri) // 2); _tr, _te = _ri[:_h], _ri[_h:2 * _h]
_lrf = LogisticRegression(C=1.0, max_iter=5000).fit(X_ALL[_tr], Y_ALL[_tr])
_lrs = LogisticRegression(C=1.0, max_iter=5000).fit(X_ALL[_tr][:, S_IDX], Y_ALL[_tr])
print(f"held-out log-loss: full logistic {log_loss(Y_ALL[_te], _lrf.predict_proba(X_ALL[_te])[:, 1]):.4f} | "
      f"short score {log_loss(Y_ALL[_te], _lrs.predict_proba(X_ALL[_te][:, S_IDX])[:, 1]):.4f} | "
      f"reference XGBoost {log_loss(Y_ALL[_te], ETA_HAT[_te]):.4f}")

## 3. Helpers (same as v6)

In [ ]:
import torch, statsmodels.api as sm
from scipy import stats
from sklearn.tree import DecisionTreeRegressor
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, log_loss

COLS = FEATURES + ["y"]

def cart_synth(df, n, seed):
    rng = np.random.default_rng(seed)
    out = pd.DataFrame({COLS[0]: rng.choice(df[COLS[0]].to_numpy(), n, replace=True)})
    for j, c in enumerate(COLS[1:], 1):
        prev = COLS[:j]
        tree = DecisionTreeRegressor(min_samples_leaf=CART_MIN_LEAF, random_state=seed + j).fit(df[prev], df[c])
        lr_, ls_ = tree.apply(df[prev]), tree.apply(out[prev])
        vals = df[c].to_numpy(); new = np.empty(n)
        groups = pd.Series(np.arange(len(df))).groupby(lr_).apply(np.array)
        for leaf in np.unique(ls_):
            pos = np.where(ls_ == leaf)[0]
            new[pos] = vals[rng.choice(groups[leaf], len(pos), replace=True)]
        out[c] = new
    return out

def _tilt_features(X):
    # features for the density-ratio classifier: all covariates plus squares of the non-binary ones
    nb = [j for j in range(X.shape[1]) if len(np.unique(X_ALL[:2000, j])) > 2]
    return np.c_[X, X[:, nb] ** 2]

def target_resample(pool_X, local_X, n, rng):
    # Targeting done by the RECEIVER on the synthetic pool it received (its own covariates never leave the site):
    # a probabilistic classifier separates local covariates from the pool, and its odds estimate the density ratio p_R / q
    F_pool, F_loc = _tilt_features(pool_X), _tilt_features(local_X)
    clf = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(C=1.0, max_iter=5000, class_weight="balanced"))])
    clf.fit(np.vstack([F_loc, F_pool]), np.r_[np.ones(len(F_loc)), np.zeros(len(F_pool))])
    p = np.clip(clf.predict_proba(F_pool)[:, 1], 1e-6, 1 - 1e-6)
    w = p / (1 - p); w = np.minimum(w, np.quantile(w, W_CLIP_Q)); w /= w.sum()
    return rng.choice(len(pool_X), size=n, replace=True, p=w), 1 / np.sum(w ** 2) / len(w)

def fit_lr(X_hard=None, y_hard=None, X_soft=None, p_soft=None, C=1.0):
    # logistic regression on hard local labels and/or soft (teacher-probability) labels: each soft row is
    # duplicated with y=1 (weight p) and y=0 (weight 1-p), which is the infinite-relabelling limit of method B
    Xs, ys, ws = [], [], []
    if X_hard is not None:
        Xs.append(X_hard); ys.append(y_hard); ws.append(np.ones(len(y_hard)))
    if X_soft is not None:
        Xs += [X_soft, X_soft]; ys += [np.ones(len(p_soft)), np.zeros(len(p_soft))]; ws += [p_soft, 1 - p_soft]
    m = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(C=C, max_iter=5000))])
    return m.fit(np.vstack(Xs), np.concatenate(ys), lr__sample_weight=np.concatenate(ws))

def logit(p):
    p = np.clip(p, 1e-9, 1 - 1e-9); return np.log(p / (1 - p))

def _glm(y, X, offset=None, w=None):
    try:
        r = sm.GLM(y, X, family=sm.families.Binomial(), offset=offset, freq_weights=w).fit()
        if not r.converged or not np.all(np.isfinite(r.params)): return None
        return r
    except Exception:
        return None

def recal(lp_tr, y_tr, lp_te):
    r = _glm(y_tr, sm.add_constant(lp_tr, has_constant="add"))
    return expit(lp_te) if r is None else expit(r.params[0] + r.params[1] * lp_te)

def closed_test(lp_tr, y_tr, lp_te):
    # Closed testing (Vergouwe et al. 2017) among original / recalibration-in-the-large / logistic recalibration.
    # Model revision (re-estimating all coefficients) is not offered: with a few hundred patients it overfits badly.
    ll0 = np.sum(y_tr * np.log(expit(lp_tr)) + (1 - y_tr) * np.log(1 - expit(lp_tr)))
    m1 = _glm(y_tr, np.ones((len(y_tr), 1)), offset=lp_tr)
    m2 = _glm(y_tr, sm.add_constant(lp_tr, has_constant="add"))
    ll = lambda m: -np.inf if m is None else m.llf
    sig = lambda a, b, df: 2 * (ll(a) - b) > stats.chi2.ppf(1 - ALPHA_CT, df)
    if m2 is None or not sig(m2, ll0, 2):
        return expit(lp_te), "original"
    if sig(m2, ll(m1), 1):
        return expit(m2.params[0] + m2.params[1] * lp_te), "recalibration"
    return (expit(m1.params[0] + lp_te), "intercept") if m1 is not None else (expit(lp_te), "original")

def kl_excess(eta, p):
    p = np.clip(p, 1e-9, 1 - 1e-9); e = np.clip(eta, 1e-12, 1 - 1e-12)
    return float(np.mean(e * np.log(e / p) + (1 - e) * np.log((1 - e) / (1 - p))))

def teacher_fit(X, y, seed):
    xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.9,
                        eval_metric="logloss", random_state=seed, n_jobs=-1, verbosity=0)
    return CalibratedClassifierCV(xgb, method="sigmoid", cv=5).fit(X, y)



# ---------------- v8 additions ----------------
T_NB = [0.10, 0.15, 0.20]          # risk thresholds for net benefit
B_CT = 30                          # simulations of the closed test with n local patients (finite-sample prediction)

def ratio_classifier(pool_X, local_X):
    # receiver-side density-ratio model: local covariates (1) vs the donor's synthetic covariates (0)
    F_pool, F_loc = _tilt_features(pool_X), _tilt_features(local_X)
    clf = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(C=1.0, max_iter=5000, class_weight="balanced"))])
    return clf.fit(np.vstack([F_loc, F_pool]), np.r_[np.ones(len(F_loc)), np.zeros(len(F_pool))])

def est_weights(clf, X):
    p = np.clip(clf.predict_proba(_tilt_features(X))[:, 1], 1e-6, 1 - 1e-6)
    w = p / (1 - p); w = np.minimum(w, np.quantile(w, W_CLIP_Q)); return w / w.mean()

def exact_weights(X, sk, rule):
    # the true density ratio p_R / p_donor of the simulated sites (known by design): sk/(1-sk) above the median, (1-sk)/sk below
    j, thr, ge = rule
    h = X[:, j] >= thr if ge else X[:, j] > thr
    return np.where(h, sk / (1 - sk), (1 - sk) / sk)

def resample_w(w, n, rng):
    p = w / w.sum(); return rng.choice(len(w), size=n, replace=True, p=p), 1 / np.sum(p ** 2) / len(p)

def fit_lr_w(X, y, w):
    m = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(C=1.0, max_iter=5000))])
    return m.fit(X, y, lr__sample_weight=w)

def net_benefit(y, p, t):
    pos = p >= t
    return float(np.mean(pos & (y == 1)) - np.mean(pos & (y == 0)) * t / (1 - t))

def cal_stats(y, p):
    lp = logit(p)
    s = _glm(y, sm.add_constant(lp, has_constant="add")); c = _glm(y, np.ones((len(y), 1)), offset=lp)
    return (np.nan if s is None else s.params[1]), (np.nan if c is None else c.params[0])

def ct_finite(lp_fn, X_pop, eta_pop, lp_te, e_te, n, B, rng):
    # expected excess risk of "population donor model + closed test on n local patients": the finite-sample
    # counterpart of Gamma_upd (the asymptotic version assumes perfect local updating)
    out = []
    for _ in range(B):
        i = rng.choice(len(X_pop), n, replace=False)
        yb = (rng.random(n) < eta_pop[i]).astype(int)
        p, _ = closed_test(lp_fn(X_pop[i]), yb, lp_te)
        out.append(kl_excess(e_te, p))
    return float(np.mean(out))

def draw(avail, n, share_high, high, rng):
    nh = int(round(share_high * n))
    idx = np.r_[rng.choice(np.flatnonzero(avail & high), nh, replace=False),
                rng.choice(np.flatnonzero(avail & ~high), n - nh, replace=False)]
    avail[idx] = False
    return rng.permutation(idx)                     # shuffle: draws are built high-then-low, and calibration CV folds must not follow that order

def proj_score(X, p):
    # population I-projection of probabilities p onto the short logistic score
    return fit_lr(X_soft=X[:, S_IDX], p_soft=p, C=1e4)
def _coef_rows(models, Xte, cols, meta):
    sd = Xte.std(0); sd[sd == 0] = 1; mad = np.abs(Xte - Xte.mean(0)).mean(0); out = []
    for name, m in models.items():
        sc, lr = m.named_steps["sc"], m.named_steps["lr"]
        b = lr.coef_[0] / np.where(sc.scale_ == 0, 1, sc.scale_)
        for j, c in enumerate(cols):
            out.append(dict(meta, model=name, feature=c, beta_std=b[j] * sd[j], imp=abs(b[j]) * mad[j]))
    return out


## 4. Run (resumes from the checkpoint file)

In [ ]:
COLUMNS = ["dataset", "shift", "share", "rep", "arm", "logloss_y", "brier_y", "auc", "ex_kl_ref", "ct_choice", "ess_C", "cal_slope", "citl", "nb10", "nb15", "nb20", "est_G", "est_Gupd_n", "ll_teacher_loc", "ll_score_loc", "ess_don", "p_better", "d_mean", "est_null"]

def done_keys():
    if not os.path.exists(OUT_CSV): return set()
    r = pd.read_csv(OUT_CSV); return set(zip(r["shift"], r["share"], r["rep"]))

def run_cell(shift, sk, rep):
    t0 = time.time()
    seed = SEED0 + 1000 * rep + 100 * list(SHIFTS).index(shift) + 10 * SKEWS.index(sk)
    rng = np.random.default_rng(seed)
    high = HIGH[shift]
    avail = ~IN_REF.copy()
    te, loc, rpop = draw(avail, TEST_N, sk, high, rng), draw(avail, N_LOCAL, sk, high, rng), draw(avail, POP_N, sk, high, rng)
    don, dpop = draw(avail, N_DONOR, 1 - sk, high, rng), draw(avail, POP_N, 1 - sk, high, rng)
    X, y, S = X_ALL, Y_ALL, S_IDX; SIDX = S
    res = {}

    res["local_score"] = fit_lr(X[loc][:, S], y[loc]).predict_proba(X[te][:, S])[:, 1]
    star, dstar = proj_score(X[rpop], ETA_HAT[rpop]), proj_score(X[dpop], ETA_HAT[dpop])
    res["oracle_star"] = star.predict_proba(X[te][:, S])[:, 1]
    res["oracle_D"] = dstar.predict_proba(X[te][:, S])[:, 1]
    rc = fit_lr(X_soft=dstar.decision_function(X[rpop][:, S])[:, None], p_soft=ETA_HAT[rpop], C=1e4)
    res["oracle_D_recal"] = rc.predict_proba(dstar.decision_function(X[te][:, S])[:, None])[:, 1]

    hD = fit_lr(X[don][:, S], y[don])                                  # the donor publishes its short score
    lp_loc, lp_te = hD.decision_function(X[loc][:, S]), hD.decision_function(X[te][:, S])
    res["share_score"] = expit(lp_te)
    res["share_score_recal"] = recal(lp_loc, y[loc], lp_te)
    res["share_score_ct"], ct = closed_test(lp_loc, y[loc], lp_te)
    lpf = lambda Z: dstar.decision_function(Z[:, S])
    extra = {"oracle_D_ct_n": ct_finite(lpf, X[rpop], ETA_HAT[rpop], lpf(X[te]), ETA_HAT[te], N_LOCAL, B_CT, rng)}

    _tsub = don if TEACHER_N is None else np.random.default_rng(seed + 5).choice(don, TEACHER_N, replace=False)
    t = teacher_fit(X[_tsub], y[_tsub], seed)                               # teacher sees all predictors
    res["teacher_full"] = t.predict_proba(X[te])[:, 1]
    ddf = pd.DataFrame(X[don], columns=FEATURES).assign(y=y[don])
    pool = cart_synth(ddf, POOL_MULT * N_DONOR, seed)[FEATURES].to_numpy(float)
    p_pool = t.predict_proba(pool)[:, 1]
    clf = ratio_classifier(pool, X[loc])                              # targeting uses all predictors held locally
    idxC, ess = resample_w(est_weights(clf, pool), N_SYN, rng)
    idxE, _ = resample_w(exact_weights(pool, sk, RULES[shift]), N_SYN, rng)
    hIW = fit_lr_w(X[don][:, S], y[don], est_weights(clf, X[don]))
    res["IW_donor"] = hIW.predict_proba(X[te][:, S])[:, 1]
    res["IW_donor_ct"], _ = closed_test(hIW.decision_function(X[loc][:, S]), y[loc], hIW.decision_function(X[te][:, S]))
    res["IW_donor_exact"] = fit_lr_w(X[don][:, S], y[don], exact_weights(X[don], sk, RULES[shift])).predict_proba(X[te][:, S])[:, 1]
    res["C_exact_ratio"] = fit_lr(X[loc][:, S], y[loc], pool[idxE][:, S], p_pool[idxE]).predict_proba(X[te][:, S])[:, 1]
    # ---- v9: what the receiver can estimate BEFORE choosing (no access to the truth) ----
    Xt, pt = pool[idxC], p_pool[idxC]                                   # targeted pool with teacher soft labels
    h_hat = fit_lr(X_soft=Xt[:, S], p_soft=pt, C=1e4)                    # receiver's estimate of its best simple score
    ph = h_hat.predict_proba(Xt[:, S])[:, 1]
    lpD_t = hD.decision_function(Xt[:, S])
    est_G = kl_excess(ph, expit(lpD_t))                                 # estimated gain over the shared score
    upd = []
    for _ in range(B_CT):                                               # closed test on n simulated local patients
        i = rng.choice(len(Xt), N_LOCAL, replace=False)
        yb = (rng.random(N_LOCAL) < pt[i]).astype(int)
        p_b, _ = closed_test(lpD_t[i], yb, lpD_t)
        upd.append(kl_excess(ph, p_b))
    est_Gupd_n = float(np.mean(upd))                                    # estimated gain over the updated shared score
    ll_teacher_loc = log_loss(y[loc], np.clip(t.predict_proba(X[loc])[:, 1], 1e-9, 1 - 1e-9), labels=[0, 1])
    ll_score_loc = log_loss(y[loc], np.clip(expit(lp_loc), 1e-9, 1 - 1e-9), labels=[0, 1])
    w_d = est_weights(clf, X[don]); ess_don = float(w_d.sum() ** 2 / np.sum(w_d ** 2) / len(w_d))

    # ---- v13: bootstrap of Q1, bias-corrected gain estimate, absolute ESS ----
    rng2 = np.random.default_rng(seed + 999)
    _lt = np.clip(t.predict_proba(X[loc])[:, 1], 1e-9, 1 - 1e-9); _ls = np.clip(expit(lp_loc), 1e-9, 1 - 1e-9); _yy = y[loc]
    _d = -(_yy * np.log(_lt) + (1 - _yy) * np.log(1 - _lt)) + (_yy * np.log(_ls) + (1 - _yy) * np.log(1 - _ls))
    _bm = _d[rng2.integers(0, len(_d), (2000, len(_d)))].mean(1)
    p_better, d_mean = float((_bm < 0).mean()), float(_d.mean())
    Xu, pu = pool[np.arange(N_SYN)], p_pool[np.arange(N_SYN)]           # untargeted, donor-like pool: true gain there is ~0
    h_u = fit_lr(X_soft=Xu[:, SIDX], p_soft=pu, C=1e4); ph_u = h_u.predict_proba(Xu[:, SIDX])[:, 1]
    lpD_u = hD.decision_function(Xu[:, SIDX]); upd_u = []
    for _ in range(B_CT):
        i = rng2.choice(len(Xu), N_LOCAL, replace=False)
        yb = (rng2.random(N_LOCAL) < pu[i]).astype(int)
        p_b, _ = closed_test(lpD_u[i], yb, lpD_u)
        upd_u.append(kl_excess(ph_u, p_b))
    est_null = float(np.mean(upd_u))

    B = np.arange(N_SYN)
    mB = fit_lr(X[loc][:, S], y[loc], pool[B][:, S], p_pool[B]); res["B_untargeted"] = mB.predict_proba(X[te][:, S])[:, 1]
    mC = fit_lr(X[loc][:, S], y[loc], pool[idxC][:, S], p_pool[idxC]); res["C_target"] = mC.predict_proba(X[te][:, S])[:, 1]
    res["local_distill"] = fit_lr(X_soft=X[loc][:, S], p_soft=t.predict_proba(X[loc])[:, 1]).predict_proba(X[te][:, S])[:, 1]

    rows = []
    for arm, p in res.items():
        pc = np.clip(p, 1e-9, 1 - 1e-9)
        rows.append(dict(dataset=DATASET, shift=shift, share=sk, rep=rep, arm=arm,
                         logloss_y=log_loss(y[te], pc, labels=[0, 1]), brier_y=float(np.mean((pc - y[te]) ** 2)),
                         auc=roc_auc_score(y[te], pc), ex_kl_ref=kl_excess(ETA_HAT[te], pc),
                         ct_choice=ct if arm == "share_score_ct" else "", ess_C=ess if arm == "C_target" else np.nan,
                         **dict(zip(["cal_slope", "citl"], cal_stats(y[te], pc))),
                         **{f"nb{int(t*100)}": net_benefit(y[te], pc, t) for t in T_NB}))
    for arm, v in extra.items():
        rows.append(dict(dataset=DATASET, shift=shift, share=sk, rep=rep, arm=arm, ex_kl_ref=v))
    rows.append(dict(dataset=DATASET, shift=shift, share=sk, rep=rep, arm="decision_inputs", est_G=est_G, est_Gupd_n=est_Gupd_n, ll_teacher_loc=ll_teacher_loc, ll_score_loc=ll_score_loc, ess_don=ess_don, p_better=p_better, d_mean=d_mean, est_null=est_null))
    _cr = _coef_rows({"shared": hD, "M2_untargeted": mB, "M3_targeted": mC, "IW": hIW, "oracle": star}, X[te][:, S], SCORE, dict(dataset=DATASET, shift=shift, share=sk, rep=rep))
    pd.DataFrame(_cr).to_csv(OUT_COEF, mode="a", header=not os.path.exists(OUT_COEF), index=False)
    pd.DataFrame(rows).reindex(columns=COLUMNS).to_csv(OUT_CSV, mode="a", header=not os.path.exists(OUT_CSV), index=False)
    print(f"  {shift:26s} share {sk:.1f} rep {rep:2d}  {time.time() - t0:5.1f}s")

t_start = time.time()
for rep in range(N_REPS):
    _done = done_keys()
    for shift, sk in SCEN:
        if (shift, sk, rep) not in _done: run_cell(shift, sk, rep)
print(f"Finished in {(time.time() - t_start) / 60:.1f} min -> {OUT_CSV}")

## Results: uncertainty in Q1, ESS threshold, gain prediction, held-out evaluation, ablation, explainability

In [ ]:

import matplotlib.pyplot as plt
from scipy.stats import spearmanr
R = pd.read_csv(OUT_CSV); CO = pd.read_csv(OUT_COEF)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60)
CELL = ["shift", "share"]; M, MR = "logloss_y", "ex_kl_ref"
A = dict(loc="local_score", ct="share_score_ct", raw="share_score", teach="teacher_full", star="oracle_star")
split = lambda key: "design" if tuple(key) in DESIGN else "held-out"
P = R.pivot_table(index=CELL + ["rep"], columns="arm", values=M) * 1000
PR = R.pivot_table(index=CELL + ["rep"], columns="arm", values=MR) * 1000
I = R[R.arm == "decision_inputs"].set_index(CELL + ["rep"])
J = pd.DataFrame({"M0": P[A["loc"]], "M1": P[A["ct"]], "M2": P["B_untargeted"], "M3": P["C_target"], "IW": P["IW_donor"],
                  "obs_gain": P[A["ct"]] - P["C_target"], "pred_n": PR["oracle_D_ct_n"] - PR[A["star"]],
                  "teacher_true_better": P[A["teach"]] < P[A["raw"]]}).join(I[["p_better", "d_mean", "est_Gupd_n", "est_null", "ess_don"]])
J["est_raw"] = J.est_Gupd_n * 1000; J["est_corr"] = (J.est_Gupd_n - J.est_null) * 1000
J["split"] = [split(k[:len(CELL)]) for k in J.index]

# ---------- (3) predicted vs observed gain, all scenarios ----------
C = J.groupby(level=CELL)[["obs_gain", "pred_n", "est_raw", "est_corr"]].mean()
C["split"] = [split(k if isinstance(k, tuple) else (k,)) for k in C.index]
def acc(p, o):
    e = p - o; return pd.Series({"MAE": e.abs().mean(), "RMSE": np.sqrt((e ** 2).mean()), "bias": e.mean(), "Spearman": spearmanr(p, o)[0]})
T3 = pd.DataFrame({f"{lab} ({s})": acc(C.loc[m_, col], C.loc[m_, "obs_gain"])
                   for col, lab in [("pred_n", "framework"), ("est_raw", "practical raw"), ("est_corr", "practical corrected")]
                   for s, m_ in [("all", C.index == C.index), ("held-out", C.split == "held-out")]}).T.round(2)
print("Predicted vs observed gain over M1 (x1000), one point per scenario"); display(T3)
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a_, col, lab in zip(ax, ["pred_n", "est_raw", "est_corr"], ["framework (true risk)", "practical estimate, raw", "practical estimate, bias-corrected"]):
    for s, mk in [("design", "o"), ("held-out", "^")]:
        d_ = C[C.split == s]; a_.scatter(d_[col], d_.obs_gain, marker=mk, label=s, s=30)
    lim = [min(C[col].min(), C.obs_gain.min()) - 3, max(C[col].max(), C.obs_gain.max()) + 3]
    a_.plot(lim, lim, color="#999", lw=0.8); a_.set_xlabel("predicted gain"); a_.set_title(lab, fontsize=10)
ax[0].set_ylabel("observed gain of M3 over M1"); ax[0].legend()
plt.tight_layout(); plt.savefig(OUT_CSV.replace(".csv", "_pred_vs_obs.png"), dpi=150); plt.show()
H131 = abs(T3.loc["practical corrected (all)", "bias"]) < abs(T3.loc["practical raw (all)", "bias"]) and T3.loc["practical corrected (all)", "MAE"] < T3.loc["practical raw (all)", "MAE"]

# ---------- (1) Q1 with uncertainty ----------
CS = {"point": None, "0.80": 0.80, "0.90": 0.90, "0.95": 0.95}
def q1(r, c): return (r.d_mean < 0) if c is None else (r.p_better >= c)
rows = []
for k, c in CS.items():
    call = J.apply(lambda r: q1(r, c), axis=1); t_ = J.teacher_true_better
    rows.append(dict(c=k, accuracy=(call == t_).mean(), false_teacher_better=(call & ~t_).sum() / max((~t_).sum(), 1),
                     missed_teacher_better=(~call & t_).sum() / max(t_.sum(), 1), share_called_better=call.mean()))
TQ = pd.DataFrame(rows).round(3); print("\nQ1 against the truth from the test set"); display(TQ)

# ---------- (2)+(4) rule family: choose on design, evaluate on held-out ----------
CANDS = ["M1", "IW", "M3"]; BEST = J[CANDS].min(axis=1)
def choose(r, c, q2, t):
    if not q1(r, c): return "M1"
    if q2 == "raw" and r.est_raw < 2: return "M1"
    if q2 == "corrected" and r.est_corr < 2: return "M1"
    return "M3" if r.ess_don < t else "IW"
def evaluate(c, q2, t, mask):
    ch = J[mask].apply(lambda r: choose(r, CS[c], q2, t), axis=1)
    val = np.array([J.loc[i, k] for i, k in ch.items()])
    return (val - BEST[mask].values).mean(), (ch.values == J[mask][CANDS].idxmin(axis=1).values).mean()
grid = [(c, q2, t) for c in CS for q2 in ["none", "raw", "corrected"] for t in [0.10, 0.20, 0.30, 0.40, 0.50]]
D, H = (J.split == "design").values, (J.split == "held-out").values
G = pd.DataFrame([dict(c=c, q2=q2, ess=t, regret_design=evaluate(c, q2, t, D)[0], regret_heldout=evaluate(c, q2, t, H)[0],
                       pick_best_heldout=evaluate(c, q2, t, H)[1]) for c, q2, t in grid])
# pre-registered tie-break: among settings within 0.005 of the best design regret, prefer no gain question, then the simplest Q1,
# then the median of the tied ESS thresholds (never an extreme of the grid by accident)
T_ = G[G.regret_design <= G.regret_design.min() + 0.005].copy()
T_["o_q2"] = T_.q2.map({"none": 0, "corrected": 1, "raw": 2}); T_["o_c"] = T_.c.map({"point": 0, "0.80": 1, "0.90": 2, "0.95": 3})
first = T_.sort_values(["o_q2", "o_c"]).iloc[0]
tied = sorted(T_[(T_.q2 == first.q2) & (T_.c == first.c)].ess)
sel = G[(G.c == first.c) & (G.q2 == first.q2) & (G.ess == tied[len(tied) // 2])].iloc[0]
v9 = G[(G.c == "point") & (G.q2 == "raw") & (G.ess == 0.30)].iloc[0]
fixed = {k: (J[H][k] - BEST[H]).mean() for k in CANDS}
print("\nRule grid (regret x1000; lower is better). Sorted by design regret:"); display(G.sort_values("regret_design").head(12).round(3))
print(f"\nSelected on design scenarios: c={sel.c}, gain question={sel.q2}, ESS<{sel.ess:.0%}"
      f"\n  regret design {sel.regret_design:.2f} -> held-out {sel.regret_heldout:.2f} (picks the best strategy in {sel.pick_best_heldout:.0%} of held-out cases)"
      f"\n  v9 rule (point, raw, 30%) held-out regret {v9.regret_heldout:.2f}"
      f"\n  best possible parameters on held-out (optimistic, for reference) {G.regret_heldout.min():.2f}"
      f"\n  fixed strategies on held-out: " + ", ".join(f"always {k} {v:.2f}" for k, v in fixed.items()))
H133 = sel.regret_heldout < min(fixed.values())
print("\nESS threshold sensitivity (selected c and q2):")
display(G[(G.c == sel.c) & (G.q2 == sel.q2)][["ess", "regret_design", "regret_heldout"]].round(3))

# ---------- (5) ablation ----------
J["M4"] = [J.loc[i, choose(r, CS[sel.c], sel.q2, sel.ess)] for i, r in J.iterrows()]
def abl(mask):
    out = {}
    for k in ["M0", "M1", "M2", "M3", "M4"]:
        g = (J[mask]["M1"] - J[mask][k]); gc = g.groupby(level=CELL).mean()
        out[k] = f"{g.mean():+.1f} (worst {gc.min():+.1f}, best {gc.max():+.1f})"
    return out
TA = pd.DataFrame({"design": abl(D), "held-out": abl(H)}); print("\nAblation: gain over M1 (x1000), mean over scenarios (worst / best scenario)"); display(TA)
gH = {k: (J[H]["M1"] - J[H][k]).groupby(level=CELL).mean() for k in ["M1", "M2", "M3", "M4"]}
H134 = gH["M4"].mean() >= max(gH[k].mean() for k in ["M1", "M2", "M3"]) and gH["M4"].min() >= max(gH[k].min() for k in ["M1", "M2", "M3"])

# ---------- (6) explainability (supplementary) ----------
key = CELL + ["rep"]
W = CO.pivot_table(index=key + ["model"], columns="feature", values="beta_std")
IM = CO.pivot_table(index=key + ["model"], columns="feature", values="imp")
def cos(a, b): return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))
ex = []
for k_ in W.index.droplevel("model").unique():
    o = W.loc[k_ + ("oracle",)].values; oi = IM.loc[k_ + ("oracle",)].values
    row = dict(zip(key, k_))
    for m_ in ["shared", "M2_untargeted", "M3_targeted", "IW"]:
        row[f"cos_{m_}"] = cos(W.loc[k_ + (m_,)].values, o); row[f"rank_{m_}"] = spearmanr(IM.loc[k_ + (m_,)].values, oi)[0]
    ex.append(row)
EX = pd.DataFrame(ex).set_index(key).join(J[["pred_n"]])
EX["gain_cell"] = EX.groupby(level=CELL).pred_n.transform("mean") >= 2
TE = EX.groupby("gain_cell")[[c for c in EX.columns if c.startswith(("cos_", "rank_"))]].mean().round(3)
print("\nExplainability: agreement with the receiver's best simple score (cosine of standardized coefficients; Spearman of |SHAP| importance)"); display(TE)
st = []
for k_, sub in CO[CO.model.isin(["shared", "M3_targeted"])].groupby(CELL + ["model"]):
    V = sub.pivot_table(index="rep", columns="feature", values="imp").values
    rc = [spearmanr(V[i], V[j])[0] for i in range(len(V)) for j in range(i + 1, len(V))]
    st.append(dict(zip(CELL + ["model"], k_), stability=np.nanmean(rc)))
print("Importance-ranking stability across repetitions (mean pairwise Spearman):")
display(pd.DataFrame(st).groupby("model").stability.describe()[["mean", "min", "max"]].round(3))
H135 = bool(TE.loc[True, "cos_M3_targeted"] > TE.loc[True, "cos_shared"] and TE.loc[True, "rank_M3_targeted"] > TE.loc[True, "rank_shared"]) if True in TE.index else None
b1, b2 = TQ.set_index("c").loc["point"], TQ.set_index("c").loc[["0.80", "0.90"]]
H132 = bool((b2.false_teacher_better < b1.false_teacher_better).any())
print(f"\nH13.1 corrected estimate less biased: {H131}\nH13.2 bootstrap Q1 fewer false calls: {H132}\nH13.3 held-out regret below fixed strategies: {H133}"
      f"\nH13.4 M4 best mean and worst case on held-out: {H134}\nH13.5 M3 closer to the receiver's best score where a gain is predicted: {H135}")
